# Recompute 60 Connectivity Features Theo Paper

Notebook này tính lại đầy đủ **60 connectivity feature sets** từ `Cleaned_Epochs` bằng công thức trong `src/ftd_mlflow_pipeline/connectivity.py`.

- 5 frequency bands: `delta`, `theta`, `alpha`, `beta`, `gamma`
- 12 connectivity metrics: `cov`, `corr`, `xcov`, `xcorr`, `csd`, `coh`, `mi`, `ecc`, `aecov`, `aecorr`, `plv`, `wplv`
- Tổng cộng: `5 x 12 = 60` ma trận feature sets

Output mặc định được lưu vào `Full_MultiDomain_Features_Role3_v4` để không ghi đè folder cũ.

## 1. Import thư viện và module project

Cell này nạp các thư viện cần thiết và import hàm `compute_feature_set` đã sửa công thức connectivity.

In [1]:
from pathlib import Path
import shutil
import sys
import time

import numpy as np
import pandas as pd

ROOT = Path('/home/dohaidang/DataMining_Project')
sys.path.insert(0, str(ROOT / 'src'))

from ftd_mlflow_pipeline.connectivity import BANDS, METRICS, CACHE_VERSION, compute_feature_set

print('Project root:', ROOT)
print('Cache version:', CACHE_VERSION)
print('Bands:', BANDS)
print('Metrics:', METRICS)
print('Total feature sets:', len(BANDS) * len(METRICS))

Project root: /home/dohaidang/DataMining_Project
Cache version: v4
Bands: ('delta', 'theta', 'alpha', 'beta', 'gamma')
Metrics: ('cov', 'corr', 'xcov', 'xcorr', 'csd', 'coh', 'mi', 'ecc', 'aecov', 'aecorr', 'plv', 'wplv')
Total feature sets: 60


## 2. Cấu hình đường dẫn

`CLEANED_EPOCHS_DIR` là dữ liệu đầu vào. `OUTPUT_DIR` là nơi lưu 60 file `.npy` mới.

Nếu muốn tính lại từ đầu bất kể cache có sẵn, đặt `FORCE_RECOMPUTE = True`.

In [2]:
CLEANED_EPOCHS_DIR = ROOT / 'Cleaned_Epochs'
CACHE_DIR = ROOT / '.cache' / 'connectivity'
OUTPUT_DIR = ROOT / 'Full_MultiDomain_Features_Role3_v4'

FORCE_RECOMPUTE = False
SKIP_EXISTING_OUTPUT = True
SAVE_FULL_NPZ = False

if not CLEANED_EPOCHS_DIR.exists():
    raise FileNotFoundError(f'Khong tim thay folder Cleaned_Epochs: {CLEANED_EPOCHS_DIR}')

OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('Input:', CLEANED_EPOCHS_DIR)
print('Cache:', CACHE_DIR)
print('Output:', OUTPUT_DIR)
print('FORCE_RECOMPUTE:', FORCE_RECOMPUTE)
print('SKIP_EXISTING_OUTPUT:', SKIP_EXISTING_OUTPUT)

Input: /home/dohaidang/DataMining_Project/Cleaned_Epochs
Cache: /home/dohaidang/DataMining_Project/.cache/connectivity
Output: /home/dohaidang/DataMining_Project/Full_MultiDomain_Features_Role3_v4
FORCE_RECOMPUTE: False
SKIP_EXISTING_OUTPUT: True


## 3. Kiểm tra dữ liệu đầu vào

Mỗi subject cần có đủ 5 file epoch tương ứng 5 dải tần. Cell này chỉ kiểm tra số lượng file và một vài tên mẫu.

In [3]:
epoch_files = sorted(CLEANED_EPOCHS_DIR.glob('*-epo.fif'))
print('Total epoch files:', len(epoch_files))
print('First 5 files:')
for path in epoch_files[:5]:
    print(' -', path.name)

band_counts = {band: len(list(CLEANED_EPOCHS_DIR.glob(f'*_{band}-epo.fif'))) for band in BANDS}
pd.Series(band_counts, name='n_files').to_frame()

Total epoch files: 440
First 5 files:
 - sub-001_A_alpha-epo.fif
 - sub-001_A_beta-epo.fif
 - sub-001_A_delta-epo.fif
 - sub-001_A_gamma-epo.fif
 - sub-001_A_theta-epo.fif


,n_files
delta,88
theta,88
alpha,88
beta,88
gamma,88


## 4. Hàm lưu feature set

Mỗi feature set được lưu thành một file `.npy` riêng, ví dụ `delta_corr.npy`. Label và subject id được lưu một lần ở cuối, sau khi kiểm tra alignment.

In [4]:
def feature_output_path(band: str, metric: str) -> Path:
    return OUTPUT_DIR / f'{band}_{metric}.npy'


def cache_path_for(band: str, metric: str) -> Path:
    return CACHE_DIR / f'{CACHE_VERSION}_{band}_{metric}.npz'


def remove_cache_if_needed(band: str, metric: str) -> None:
    if FORCE_RECOMPUTE:
        path = cache_path_for(band, metric)
        if path.exists():
            path.unlink()


def save_feature_set(feature_set, output_path: Path) -> dict:
    np.save(output_path, feature_set.matrices)
    return {
        'feature': f'{feature_set.band}_{feature_set.metric}',
        'band': feature_set.band,
        'metric': feature_set.metric,
        'path': str(output_path),
        'n_epochs': int(feature_set.matrices.shape[0]),
        'n_channels': int(feature_set.matrices.shape[1]),
        'n_channels_2': int(feature_set.matrices.shape[2]),
        'dtype': str(feature_set.matrices.dtype),
        'cache_version': CACHE_VERSION,
    }

## 5. Tính đủ 60 connectivity feature sets

Cell này là phần chạy chính. Thời gian có thể lâu vì mỗi tổ hợp `band + metric` phải duyệt toàn bộ epoch.

In [5]:
metadata_rows = []
reference_labels = None
reference_subject_ids = None
all_feature_names = []

total = len(BANDS) * len(METRICS)
counter = 0
start_all = time.time()

for band in BANDS:
    for metric in METRICS:
        counter += 1
        feature_name = f'{band}_{metric}'
        output_path = feature_output_path(band, metric)
        start = time.time()

        if SKIP_EXISTING_OUTPUT and output_path.exists() and not FORCE_RECOMPUTE:
            matrices = np.load(output_path, mmap_mode='r')
            metadata_rows.append({
                'feature': feature_name,
                'band': band,
                'metric': metric,
                'path': str(output_path),
                'n_epochs': int(matrices.shape[0]),
                'n_channels': int(matrices.shape[1]),
                'n_channels_2': int(matrices.shape[2]),
                'dtype': str(matrices.dtype),
                'cache_version': CACHE_VERSION,
                'seconds': time.time() - start,
                'status': 'skipped_existing_output',
            })
            all_feature_names.append(feature_name)
            print(f'[{counter:02d}/{total}] {feature_name}: skipped existing output {tuple(matrices.shape)}')
            continue

        remove_cache_if_needed(band, metric)
        feature_set = compute_feature_set(CLEANED_EPOCHS_DIR, CACHE_DIR, band, metric)

        if reference_labels is None:
            reference_labels = feature_set.group_codes.copy()
            reference_subject_ids = feature_set.subject_ids.copy()
        else:
            if not np.array_equal(reference_labels, feature_set.group_codes):
                raise ValueError(f'Label alignment mismatch at {feature_name}')
            if not np.array_equal(reference_subject_ids, feature_set.subject_ids):
                raise ValueError(f'Subject alignment mismatch at {feature_name}')

        row = save_feature_set(feature_set, output_path)
        row['seconds'] = time.time() - start
        row['status'] = 'computed'
        metadata_rows.append(row)
        all_feature_names.append(feature_name)

        print(
            f'[{counter:02d}/{total}] {feature_name}: '
            f'{feature_set.matrices.shape} | {row["seconds"]:.1f}s'
        )

elapsed = time.time() - start_all
print(f'Finished {len(all_feature_names)} feature sets in {elapsed / 60:.1f} minutes')

Loading data for 119 events and 2500 original time points ...
Loading data for 158 events and 2500 original time points ...
Loading data for 61 events and 2500 original time points ...
Loading data for 135 events and 2500 original time points ...
Loading data for 135 events and 2500 original time points ...
Loading data for 127 events and 2500 original time points ...
Loading data for 153 events and 2500 original time points ...
Loading data for 159 events and 2500 original time points ...
Loading data for 120 events and 2500 original time points ...
Loading data for 251 events and 2500 original time points ...
Loading data for 127 events and 2500 original time points ...
Loading data for 156 events and 2500 original time points ...
Loading data for 167 events and 2500 original time points ...
Loading data for 175 events and 2500 original time points ...
Loading data for 180 events and 2500 original time points ...
Loading data for 180 events and 2500 original time points ...
Loading d

KeyboardInterrupt: 

## 6. Lưu label, subject id và metadata

Các file này giúp notebook train/baseline đọc lại feature nhanh mà không cần tính lại connectivity.

In [ ]:
if reference_labels is None or reference_subject_ids is None:
    labels_path = OUTPUT_DIR / 'labels.npy'
    subject_ids_path = OUTPUT_DIR / 'subject_ids.npy'
    if labels_path.exists() and subject_ids_path.exists():
        reference_labels = np.load(labels_path, allow_pickle=True)
        reference_subject_ids = np.load(subject_ids_path, allow_pickle=True)
        print('Loaded existing labels and subject_ids because all feature outputs were skipped.')
    else:
        raise RuntimeError('Chua co feature nao duoc tinh. Kiem tra lai cell chay chinh.')

np.save(OUTPUT_DIR / 'labels.npy', reference_labels)
np.save(OUTPUT_DIR / 'subject_ids.npy', reference_subject_ids)
np.save(OUTPUT_DIR / 'all_feature_names.npy', np.asarray(all_feature_names, dtype=object))

metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_csv(OUTPUT_DIR / 'feature_metadata.csv', index=False)

print('Saved labels:', OUTPUT_DIR / 'labels.npy')
print('Saved subject ids:', OUTPUT_DIR / 'subject_ids.npy')
print('Saved feature names:', OUTPUT_DIR / 'all_feature_names.npy')
print('Saved metadata:', OUTPUT_DIR / 'feature_metadata.csv')
metadata_df.head()

## 7. Kiểm tra output đủ 60 feature

Cell này xác nhận output có đủ 60 file `.npy` feature, không tính `labels.npy`, `subject_ids.npy`, và metadata phụ.

In [ ]:
expected_features = [f'{band}_{metric}' for band in BANDS for metric in METRICS]
missing = [name for name in expected_features if not (OUTPUT_DIR / f'{name}.npy').exists()]
feature_files = [path for path in OUTPUT_DIR.glob('*.npy') if path.stem in expected_features]

print('Expected feature files:', len(expected_features))
print('Actual feature files:', len(feature_files))
print('Missing:', missing)

if missing:
    raise FileNotFoundError('Thieu feature output: ' + ', '.join(missing))

summary = metadata_df.groupby(['band', 'metric'])['feature'].count().unstack(fill_value=0)
summary = summary.reindex(index=BANDS, columns=METRICS)
summary

## 8. Tùy chọn lưu thêm file `.npz` tổng hợp

Mặc định không bật vì 60 feature sets có thể rất lớn. Nếu cần một file tổng hợp để di chuyển dữ liệu, đặt `SAVE_FULL_NPZ = True` rồi chạy lại cell này.

In [ ]:
if SAVE_FULL_NPZ:
    npz_path = ROOT / 'Full_MultiDomain_Features_Role3_v4.npz'
    payload = {
        'labels': np.load(OUTPUT_DIR / 'labels.npy', allow_pickle=True),
        'subject_ids': np.load(OUTPUT_DIR / 'subject_ids.npy', allow_pickle=True),
        'all_feature_names': np.asarray(expected_features, dtype=object),
    }
    for feature_name in expected_features:
        payload[feature_name] = np.load(OUTPUT_DIR / f'{feature_name}.npy')
    np.savez_compressed(npz_path, **payload)
    print('Saved:', npz_path)
else:
    print('SAVE_FULL_NPZ = False, bo qua buoc tao file npz tong hop.')

## 9. Gợi ý dùng cho notebook baseline

Sau khi notebook này chạy xong, các notebook train có thể đọc từ:

```python
PRECOMPUTED_DIR = ROOT / 'Full_MultiDomain_Features_Role3_v4'
```

Nếu muốn baseline full paper dùng công thức mới, đổi `PRECOMPUTED_DIR` trong notebook baseline sang folder trên.